# KMeans — Regionais de Saúde de MG (painel Regional x Ano)

Unidade de agrupamento: par **(REGIONAL DE SAÚDE, ANO)**. Uma mesma Regional pode cair em agrupamentos diferentes em anos diferentes -- isso é informação (mudança de perfil epidemiológico ao longo do tempo), não inconsistência.

Mesmas regras de decisão do notebook de DBSCAN equivalente (`DBSCAN/dbscan.ipynb`) -- montagem de painel a partir dos CSVs de taxa, janela comum, nenhum nulo imputado, RobustScaler + PCA condicional -- mas replicadas de forma INDEPENDENTE: este notebook não importa nada da pasta `DBSCAN/`, para que os dois algoritmos possam ser configurados e editados sem um afetar o outro. O que muda entre os dois é só o algoritmo de agrupamento: em vez de densidade (DBSCAN), o KMeans particiona o espaço em **k** regiões ao redor de k centróides, minimizando a inércia (soma dos quadrados das distâncias ao centróide). Isso troca a lógica de eps/MinPts por uma única escolha: **k**, feita pelo cotovelo da curva de inércia (Thorndike, 1953) -- também não fixado à mão.

Toda a lógica reutilizável está em `utilitarios_agrupamento_kmeansManual.py` (pipeline) e `graficos_agrupamento_kmeansManual.py` (visualizações), ambos nesta pasta; este notebook só configura e executa. Nenhum resultado é gravado em disco -- matriz, atributos e agrupamento ficam só como saída de célula, como no DBSCAN.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd()))


from utilitarios_agrupamento_kmeansManual import (
    COLUNA_ANO, COLUNA_REGIONAL,
    BLOCOS_DE_TAXA, PASTA_TAXA,
    LIMIAR_MAXIMO_DE_NULOS_POR_COLUNA,
    USAR_PCA, RAZAO_MAXIMA_ATRIBUTOS_POR_LINHA, VARIANCIA_EXPLICADA_ALVO,
    NUMERO_MAXIMO_DE_COMPONENTES, SEMENTE_ALEATORIA,
    USAR_JANELA_COMUM, ANO_INICIAL_COMUM, ANO_FINAL_COMUM,
    K_MINIMO, K_MAXIMO, SEMENTES_DA_BUSCA, MINIMO_DE_ELEMENTOS_POR_CLUSTER,
    montar_painel, calcular_janela_comum, restringir_a_janela_comum,
    descartar_colunas_sem_informacao_e_total, descartar_colunas_com_muitos_nulos, tratar_nulos_sem_imputar,
    padronizar_e_reduzir, preparar_projecao_2d, pesos_das_variaveis_no_componente,
    calcular_inercia_e_silhueta_por_k, escolher_k_pelo_cotovelo, busca_em_grade_k,
    executar_kmeans, resumir_agrupamento_kmeans, diagnostico_tamanho_dos_clusters,
    montar_resumo_parametros_kmeans, montar_tabela_pertencimento, montar_visao_agregada_regional_ano,
    montar_resumo_por_cluster,
    registrar_log,
)
from graficos_agrupamento_kmeansManual import (
    grafico_variancia_explicada, grafico_pesos_da_componente,
    grafico_cotovelo_inercia, grafico_silhueta_por_k, grafico_dispersao_pc1_pc2_kmeans,
    grafico_dispersao_pc1_pc2_por_ano,
    grafico_distribuicao_por_regional, grafico_distribuicao_por_ano,
    grafico_heatmap_regional_ano,
)

## 1. Leitura dos CSVs de taxa e reformatação (largo -> painel)

In [2]:
painel = montar_painel(blocos=BLOCOS_DE_TAXA, pasta_taxa=PASTA_TAXA)
painel.head()

[LOG] tabela_casos_por_faixa_etária_taxa.csv: 9 categoria(s), 1064 par(es) regional-ano.
[LOG] tabela_casos_por_escolaridade_taxa.csv: 8 categoria(s), 728 par(es) regional-ano.
[LOG] tabela_casos_por_raça_e_cor_taxa.csv: 7 categoria(s), 728 par(es) regional-ano.
[LOG] tabela_mortes_por_faixa_etária_taxa.csv: 9 categoria(s), 1288 par(es) regional-ano.
[LOG] tabela_mamografias_por_faixa_etária_taxa.csv: 9 categoria(s), 364 par(es) regional-ano.
[LOG] Painel montado: 1316 par(es) regional-ano x 42 variável(is), 28 regional(is).


casos-faixa_0-9  casos-faixa_10-19  casos-faixa_20-29  \
REGIONAL DE SAÚDE ANO                                                           
alfenas           1979              NaN                NaN                NaN   
                  1980              NaN                NaN                NaN   
                  1981              NaN                NaN                NaN   
                  1982              NaN                NaN                NaN   
                  1983              NaN                NaN                NaN   

                        casos-faixa_30-39  casos-faixa_40-49  \
REGIONAL DE SAÚDE ANO                                          
alfenas           1979                NaN                NaN   
                  1980                NaN                NaN   
                  1981                NaN                NaN   
                  1982                NaN                NaN   
                  1983                NaN                NaN   

                        casos-faixa_50-59  casos-faixa_60-69  \
REGIONAL DE SAÚDE ANO                                          
alfenas           1979                NaN                NaN   
                  1980                NaN                NaN   
                  1981                NaN                NaN   
                  1982                NaN                NaN   
                  1983                NaN                NaN   

                        casos-faixa_70-ou-mais  casos-faixa_total  \
REGIONAL DE SAÚDE ANO                                               
alfenas           1979                     NaN                NaN   
                  1980                     NaN                NaN   
                  1981                     NaN                NaN   
                  1982                     NaN                NaN   
                  1983                     NaN                NaN   

                        casos-escolaridade_fundamental-completo  ...  \
REGIONAL DE SAÚDE ANO                                            ...   
alfenas           1979                                      NaN  ...   
                  1980                                      NaN  ...   
                  1981                                      NaN  ...   
                  1982                                      NaN  ...   
                  1983                                      NaN  ...   

                        mortalidade_total  mamografia_0-9  mamografia_10-19  \
REGIONAL DE SAÚDE ANO                                                         
alfenas           1979               3.54             NaN               NaN   
                  1980               7.47             NaN               NaN   
                  1981               3.99             NaN               NaN   
                  1982               3.38             NaN               NaN   
                  1983               7.82             NaN               NaN   

                        mamografia_20-29  mamografia_30-39  mamografia_40-49  \
REGIONAL DE SAÚDE ANO                                                          
alfenas           1979               NaN               NaN               NaN   
                  1980               NaN               NaN               NaN   
                  1981               NaN               NaN               NaN   
                  1982               NaN               NaN               NaN   
                  1983               NaN               NaN               NaN   

                        mamografia_50-59  mamografia_60-69  \
REGIONAL DE SAÚDE ANO                                        
alfenas           1979               NaN               NaN   
                  1980               NaN               NaN   
                  1981               NaN               NaN   
                  1982               NaN               NaN   
                  1983               NaN               NaN   

                   

In [3]:
print(
    f"Matriz de entrada (painel bruto): {painel.shape[0]} pares Regional-Ano x {painel.shape[1]} variáveis, "
    f"{painel.index.get_level_values(COLUNA_REGIONAL).nunique()} Regionais de Saúde."
)
pd.set_option("display.max_columns", None)
painel.head(10)

Matriz de entrada (painel bruto): 1316 pares Regional-Ano x 42 variáveis, 28 Regionais de Saúde.


casos-faixa_0-9  casos-faixa_10-19  casos-faixa_20-29  \
REGIONAL DE SAÚDE ANO                                                           
alfenas           1979              NaN                NaN                NaN   
                  1980              NaN                NaN                NaN   
                  1981              NaN                NaN                NaN   
                  1982              NaN                NaN                NaN   
                  1983              NaN                NaN                NaN   
                  1984              NaN                NaN                NaN   
                  1985              NaN                NaN                NaN   
                  1986              NaN                NaN                NaN   
                  1987              NaN                NaN                NaN   
                  1988              NaN                NaN                NaN   

                        casos-faixa_30-39  casos-faixa_40-49  \
REGIONAL DE SAÚDE ANO                                          
alfenas           1979                NaN                NaN   
                  1980                NaN                NaN   
                  1981                NaN                NaN   
                  1982                NaN                NaN   
                  1983                NaN                NaN   
                  1984                NaN                NaN   
                  1985                NaN                NaN   
                  1986                NaN                NaN   
                  1987                NaN                NaN   
                  1988                NaN                NaN   

                        casos-faixa_50-59  casos-faixa_60-69  \
REGIONAL DE SAÚDE ANO                                          
alfenas           1979                NaN                NaN   
                  1980                NaN                NaN   
                  1981                NaN                NaN   
                  1982                NaN                NaN   
                  1983                NaN                NaN   
                  1984                NaN                NaN   
                  1985                NaN                NaN   
                  1986                NaN                NaN   
                  1987                NaN                NaN   
                  1988                NaN                NaN   

                        casos-faixa_70-ou-mais  casos-faixa_total  \
REGIONAL DE SAÚDE ANO                                               
alfenas           1979                     NaN                NaN   
                  1980                     NaN                NaN   
                  1981                     NaN                NaN   
                  1982                     NaN                NaN   
                  1983                     NaN                NaN   
                  1984                     NaN                NaN   
                  1985                     NaN                NaN   
                  1986                     NaN                NaN   
                  1987                     NaN                NaN   
                  1988                     NaN                NaN   

                        casos-escolaridade_fundamental-completo  \
REGIONAL DE SAÚDE ANO                                             
alfenas           1979                                      NaN   
                  1980                                      NaN   
                  1981                                      NaN   
                  1982                                      NaN   
                  1983                                      NaN   
                  1984                                      NaN   
                  1985                                      NaN   
                  1986                                      NaN   
              

In [4]:
matriz_exibicao = descartar_colunas_sem_informacao_e_total(painel).reset_index()
print(
    f"Matriz de entrada (Ano como coluna; sem colunas 'total'/'sem informação'): "
    f"{matriz_exibicao.shape[0]} linhas x {matriz_exibicao.shape[1]} colunas, "
    f"{matriz_exibicao[COLUNA_REGIONAL].nunique()} Regionais de Saúde."
)
matriz_exibicao.head(10)

[LOG] Descartada(s) 7 coluna(s) sem-informação/total: ['casos-faixa_total', 'casos-escolaridade_sem-informacao', 'casos-escolaridade_total', 'casos-raca-cor_sem-informacao', 'casos-raca-cor_total', 'mortalidade_total', 'mamografia_total']
Matriz de entrada (Ano como coluna; sem colunas 'total'/'sem informação'): 1316 linhas x 37 colunas, 28 Regionais de Saúde.


,REGIONAL DE SAÚDE,ANO,casos-faixa_0-9,casos-faixa_10-19,casos-faixa_20-29,casos-faixa_30-39,casos-faixa_40-49,casos-faixa_50-59,casos-faixa_60-69,casos-faixa_70-ou-mais,casos-escolaridade_fundamental-completo,casos-escolaridade_fundamental-incompleto,casos-escolaridade_nenhuma,casos-escolaridade_nivel-medio,casos-escolaridade_nivel-superior-completo,casos-escolaridade_nivel-superior-incompleto,casos-raca-cor_amarela,casos-raca-cor_branca,casos-raca-cor_indigena,casos-raca-cor_parda,casos-raca-cor_preta,mortalidade_0-9,mortalidade_10-19,mortalidade_20-29,mortalidade_30-39,mortalidade_40-49,mortalidade_50-59,mortalidade_60-69,mortalidade_70-ou-mais,mamografia_0-9,mamografia_10-19,mamografia_20-29,mamografia_30-39,mamografia_40-49,mamografia_50-59,mamografia_60-69,mamografia_70-ou-mais
0,alfenas,1979,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.37,28.32,14.14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,alfenas,1980,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.96,21.25,55.53,41.03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,alfenas,1981,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.89,6.94,27.26,13.27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,alfenas,1982,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.93,NaN,17.83,NaN,60.19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,alfenas,1983,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.82,19.98,8.75,50.06,76.91,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,alfenas,1984,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.05,25.76,36.43,110.58,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,alfenas,1985,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.39,42.12,11.78,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,alfenas,1986,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.71,18.78,41.32,11.44,33.85,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,alfenas,1987,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.84,6.13,NaN,33.29,64.86,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,alfenas,1988,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.91,43.07,77.70,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Janela temporal, descarte de colunas e tratamento de nulos

`USAR_JANELA_COMUM=True` restringe o painel à interseção dos períodos declarados de cada bloco (ou ao período manual `ANO_INICIAL_COMUM`/`ANO_FINAL_COMUM`, se informado); `False` usa o período completo de cada bloco, sem cortar ano nenhum.

Nos CSVs de taxa, uma célula fica em branco (NaN) quando a soma bruta da categoria-ano-regional é NaN em todos os municípios -- `calculo_taxa.ipynb` preserva essa ambiguidade de propósito (`sum(min_count=1)`): não dá para distinguir "zero eventos" de "sem população válida para calcular" só olhando a taxa. Por isso **nenhum nulo vira 0 aqui** -- mesma regra do DBSCAN, para os dois algoritmos serem comparáveis sobre a mesma matriz. A sequência abaixo só remove o que é estrutural (fora da janela, categoria sem-informação/total, atributo quase todo nulo) e descarta as linhas que ainda restarem com algum valor ausente -- sem preencher nada.

In [5]:
if USAR_JANELA_COMUM:
    janela = calcular_janela_comum(BLOCOS_DE_TAXA, ano_inicial_manual=ANO_INICIAL_COMUM, ano_final_manual=ANO_FINAL_COMUM)
    matriz = restringir_a_janela_comum(painel, janela)
else:
    registrar_log("USAR_JANELA_COMUM=False: usando o período completo de cada bloco, sem restringir ano.")
    matriz = painel

matriz = descartar_colunas_sem_informacao_e_total(matriz)
matriz = descartar_colunas_com_muitos_nulos(matriz, limiar_maximo=LIMIAR_MAXIMO_DE_NULOS_POR_COLUNA)
matriz_antes_do_dropna = matriz  # snapshot p/ o diagnóstico explícito de linhas/colunas removidas, abaixo
matriz = tratar_nulos_sem_imputar(matriz)

registrar_log(f"Matriz final: {matriz.shape[0]} linha(s) x {matriz.shape[1]} atributo(s).")
matriz.head()

[LOG] USAR_JANELA_COMUM=False: usando o período completo de cada bloco, sem restringir ano.
[LOG] Descartada(s) 7 coluna(s) sem-informação/total: ['casos-faixa_total', 'casos-escolaridade_sem-informacao', 'casos-escolaridade_total', 'casos-raca-cor_sem-informacao', 'casos-raca-cor_total', 'mortalidade_total', 'mamografia_total']
[LOG] Descartada(s) 13 coluna(s) com mais de 70% de nulos: ['casos-faixa_0-9', 'casos-escolaridade_nivel-superior-incompleto', 'mortalidade_0-9', 'mortalidade_10-19', 'mortalidade_20-29', 'mamografia_0-9', 'mamografia_10-19', 'mamografia_20-29', 'mamografia_30-39', 'mamografia_40-49', 'mamografia_50-59', 'mamografia_60-69', 'mamografia_70-ou-mais']
[LOG] Descartada(s) 945 de 1316 linha(s) (71.8%) por ter ao menos um valor ausente -- nenhum valor foi imputado.
[LOG] Matriz final: 371 linha(s) x 22 atributo(s).


casos-faixa_10-19  casos-faixa_20-29  \
REGIONAL DE SAÚDE ANO                                          
alfenas           2003               0.00               2.78   
                  2005               2.51               2.76   
                  2007               0.00               2.74   
                  2008               2.57               2.73   
                  2009               0.00               0.00   

                        casos-faixa_30-39  casos-faixa_40-49  \
REGIONAL DE SAÚDE ANO                                          
alfenas           2003              21.46              57.42   
                  2005              33.72              79.50   
                  2007              29.56              59.90   
                  2008              46.35              71.84   
                  2009              24.20             118.53   

                        casos-faixa_50-59  casos-faixa_60-69  \
REGIONAL DE SAÚDE ANO                                          
alfenas           2003              62.34              82.33   
                  2005             124.65             155.58   
                  2007              88.46             160.24   
                  2008             157.17             149.13   
                  2009             155.03             251.94   

                        casos-faixa_70-ou-mais  \
REGIONAL DE SAÚDE ANO                            
alfenas           2003                   86.69   
                  2005                   81.61   
                  2007                  164.42   
                  2008                  212.56   
                  2009                  278.01   

                        casos-escolaridade_fundamental-completo  \
REGIONAL DE SAÚDE ANO                                             
alfenas           2003                                     0.00   
                  2005                                     2.36   
                  2007                                     5.67   
                  2008                                    17.96   
                  2009                                    22.18   

                        casos-escolaridade_fundamental-incompleto  \
REGIONAL DE SAÚDE ANO                                               
alfenas           2003                                       0.94   
                  2005                                       6.14   
                  2007                                      12.77   
                  2008                                      18.90   
                  2009                                      28.78   

                        casos-escolaridade_nenhuma  \
REGIONAL DE SAÚDE ANO                                
alfenas           2003                        0.94   
                  2005                        1.42   
                  2007                        1.42   
                  2008                        2.36   
                  2009                        4.25   

                        casos-escolaridade_nivel-medio  \
REGIONAL DE SAÚDE ANO                                    
alfenas           2003                            1.41   
                  2005                            1.89   
                  2007                            0.95   
                  2008                            5.20   
                  2009                            5.66   

                        casos-escolaridade_nivel-superior-completo  \
REGIONAL DE SAÚDE ANO                                                
alfenas           2003                                        2.82   
                  2005                                        1.89   
                  2007                                        3.78   
                  2008                                        4.25   
                  2009                                        7.08   

                        casos-raca-cor_amarela  casos-raca-cor_branca  \
REGIONAL DE SAÚDE ANO 

### Diagnóstico: o que foi removido, explicitamente

A etapa acima só loga contagens. Aqui vão as identidades: quais colunas caíram (sem-informação/total + esparsas) e, de cada linha (par Regional-Ano) descartada pelo `dropna`, quais colunas específicas estavam nulas nela -- para conferir que o descarte é coerente com a cobertura real dos CSVs de origem, não um bug de junção.

In [6]:
colunas_descartadas = sorted(set(painel.columns) - set(matriz_antes_do_dropna.columns))
print(f"Colunas descartadas (sem-informação/total + >{LIMIAR_MAXIMO_DE_NULOS_POR_COLUNA:.0%} de nulos): {len(colunas_descartadas)} de {painel.shape[1]}")
display(pd.DataFrame({"COLUNA_DESCARTADA": colunas_descartadas}))

linhas_descartadas_idx = matriz_antes_do_dropna.index.difference(matriz.index)
print(f"\nLinhas (pares Regional-Ano) descartadas pelo dropna: {len(linhas_descartadas_idx)} de {len(matriz_antes_do_dropna)}")

mascara_nulos = matriz_antes_do_dropna.loc[linhas_descartadas_idx].isna()
colunas_nulas_por_linha = mascara_nulos.apply(lambda linha: ", ".join(linha.index[linha]), axis=1)
tabela_linhas_descartadas = colunas_nulas_por_linha.reset_index()
tabela_linhas_descartadas.columns = [COLUNA_REGIONAL, COLUNA_ANO, "COLUNAS_COM_NULO"]
tabela_linhas_descartadas

Colunas descartadas (sem-informação/total + >70% de nulos): 20 de 42


,COLUNA_DESCARTADA
0,casos-escolaridade_nivel-superior-incompleto
1,casos-escolaridade_sem-informacao
2,casos-escolaridade_total
3,casos-faixa_0-9
4,casos-faixa_total
5,casos-raca-cor_sem-informacao
6,casos-raca-cor_total
7,mamografia_0-9
8,mamografia_10-19
9,mamografia_20-29



Linhas (pares Regional-Ano) descartadas pelo dropna: 945 de 1316


,REGIONAL DE SAÚDE,ANO,COLUNAS_COM_NULO
0,alfenas,1979,"casos-faixa_10-19, casos-faixa_20-29, casos-fa..."
1,alfenas,1980,"casos-faixa_10-19, casos-faixa_20-29, casos-fa..."
2,alfenas,1981,"casos-faixa_10-19, casos-faixa_20-29, casos-fa..."
3,alfenas,1982,"casos-faixa_10-19, casos-faixa_20-29, casos-fa..."
4,alfenas,1983,"casos-faixa_10-19, casos-faixa_20-29, casos-fa..."
...,...,...,...
940,varginha,2006,casos-faixa_10-19
941,varginha,2013,casos-faixa_10-19
942,varginha,2022,casos-faixa_10-19
943,varginha,2024,"casos-faixa_10-19, casos-faixa_20-29, casos-fa..."


## 3. Padronização (RobustScaler) e PCA

`USAR_PCA=None` decide sozinho pela razão atributos/linhas (`RAZAO_MAXIMA_ATRIBUTOS_POR_LINHA`); `True` força PCA; `False` pula o PCA inteiramente -- em ambos os casos o critério automático é ignorado. Mesma configuração do DBSCAN.

In [7]:
matriz_padronizada, matriz_para_kmeans, pca, variancia_acumulada = padronizar_e_reduzir(
    matriz,
    razao_maxima_atributos_por_linha=RAZAO_MAXIMA_ATRIBUTOS_POR_LINHA,
    variancia_alvo=VARIANCIA_EXPLICADA_ALVO,
    numero_maximo_de_componentes=NUMERO_MAXIMO_DE_COMPONENTES,
    semente=SEMENTE_ALEATORIA,
    usar_pca=USAR_PCA,
)

[LOG] PCA aplicado: USAR_PCA=True (forçado; razão atributos/linhas = 0.059) -> 9 componente(s), 86.8% da variância acumulada.


### Diagnóstico: variância por componente e pesos das variáveis

Só roda quando o PCA foi de fato aplicado (`pca is not None`). O "scree plot" mostra a variância explicada por componente, com PC1 destacado em vermelho se ultrapassar `VARIANCIA_PC1_ALERTA` -- direção dominante isolada sugere que o componente captura porte/volume da regional, não perfil epidemiológico. Os pesos ("loadings") de PC1 mostram quais atributos originais mais influenciam esse eixo.

In [8]:
if pca is not None:
    grafico_variancia_explicada(pca, variancia_acumulada).show()
    display(pesos_das_variaveis_no_componente(pca, matriz.columns, componente=1))
    grafico_pesos_da_componente(pca, matriz.columns, componente=1).show()
else:
    registrar_log("USAR_PCA resultou em pca=None -- diagnóstico de variância/pesos não se aplica (KMeans rodou sobre os atributos originais).")

casos-raca-cor_amarela                        0.969077
casos-faixa_50-59                             0.089562
casos-escolaridade_fundamental-completo       0.083585
casos-faixa_40-49                             0.081156
casos-escolaridade_nivel-medio                0.074471
casos-escolaridade_nivel-superior-completo    0.063380
casos-faixa_60-69                             0.060815
casos-faixa_70-ou-mais                        0.058424
casos-faixa_30-39                             0.056636
casos-raca-cor_preta                          0.055573
casos-raca-cor_branca                         0.054806
casos-escolaridade_fundamental-incompleto     0.050840
casos-escolaridade_nenhuma                    0.049511
casos-faixa_20-29                             0.042053
mortalidade_70-ou-mais                        0.040464
mortalidade_40-49                             0.039769
mortalidade_50-59                             0.036583
casos-raca-cor_parda                          0.029685
mortalidad

## 4. Escolha de k (cotovelo da curva de inércia)

In [9]:
tabela_k = calcular_inercia_e_silhueta_por_k(matriz_para_kmeans, k_minimo=K_MINIMO, k_maximo=K_MAXIMO, semente=SEMENTE_ALEATORIA)
k_escolhido, indice_do_cotovelo = escolher_k_pelo_cotovelo(tabela_k)

figura_cotovelo = grafico_cotovelo_inercia(tabela_k, indice_do_cotovelo, k_escolhido)
figura_cotovelo.show()
tabela_k

[LOG] Inércia e silhueta calculadas para k = 2..15 (14 valor(es) de k).
[LOG] Cotovelo da curva de inércia na posição 3 de 14 -- k = 5


,k,inercia,silhueta
0,2,4410.315273,0.840336
1,3,3262.567635,0.238233
2,4,2849.669712,0.199683
3,5,2609.254173,0.211154
4,6,2401.842957,0.213493
5,7,2241.728546,0.220520
6,8,2104.481664,0.158368
7,9,1993.788116,0.184701
8,10,1907.551848,0.144178
9,11,1812.759248,0.161764


### Silhueta por k (validação complementar)

In [10]:
figura_silhueta = grafico_silhueta_por_k(tabela_k, k_escolhido)
figura_silhueta.show()

### Busca em grade (validação complementar)

In [11]:
tabela_da_busca = busca_em_grade_k(matriz_para_kmeans, k_minimo=K_MINIMO, k_maximo=K_MAXIMO, sementes=SEMENTES_DA_BUSCA)
tabela_da_busca.head(20)

,k,semente,inercia,silhueta
0,2,42,4410.315273,0.840336
1,3,42,3262.567635,0.238233
2,4,42,2849.669712,0.199683
3,5,42,2609.254173,0.211154
4,6,42,2401.842957,0.213493
5,7,42,2241.728546,0.220520
6,8,42,2104.481664,0.158368
7,9,42,1993.788116,0.184701
8,10,42,1907.551848,0.144178
9,11,42,1812.759248,0.161764


## 5. KMeans

In [12]:
rotulos, modelo_kmeans = executar_kmeans(matriz_para_kmeans, k_escolhido, semente=SEMENTE_ALEATORIA)
n_clusters, inercia_final, silhueta_final = resumir_agrupamento_kmeans(matriz_para_kmeans, rotulos, modelo_kmeans)

registrar_log(f"KMeans(k={k_escolhido}, random_state={SEMENTE_ALEATORIA})")
registrar_log(f"  Agrupamentos: {n_clusters}")
registrar_log(f"  Inércia (WCSS): {inercia_final:.4f}")
registrar_log(f"  Silhueta: {silhueta_final:.4f}" if not np.isnan(silhueta_final) else "  Silhueta: não aplicável (k < 2)")

_ = diagnostico_tamanho_dos_clusters(rotulos, minimo=MINIMO_DE_ELEMENTOS_POR_CLUSTER)

[LOG] KMeans(k=5, random_state=42)
[LOG]   Agrupamentos: 5
[LOG]   Inércia (WCSS): 2609.2542
[LOG]   Silhueta: 0.2112
[LOG] ATENÇÃO: 1 cluster(s) com menos de 4 elemento(s) -- {1: 3}. O KMeans aloca todo ponto a algum cluster (não existe 'ruído' como no DBSCAN); um cluster muito pequeno tende a ser um outlier isolado, não um perfil compartilhado -- considere reduzir k ou tratar esses pontos como atípicos na interpretação.


## Visualizações

In [13]:
matriz_2d, variancia_explicada_2d = preparar_projecao_2d(matriz_padronizada, matriz_para_kmeans, pca, semente=SEMENTE_ALEATORIA)

figura_dispersao = grafico_dispersao_pc1_pc2_kmeans(matriz_2d, rotulos, matriz.index, variancia_explicada_2d, k_escolhido)
figura_dispersao.show()

### Dispersão PC1 x PC2 por ano

Mesmos pontos e paleta do gráfico acima, mas em uma grade de mini-gráficos -- um por ano -- para comparar a composição dos agrupamentos ano a ano em vez de ver todos os pontos misturados.

In [14]:
figura_dispersao_por_ano = grafico_dispersao_pc1_pc2_por_ano(matriz_2d, rotulos, matriz.index, variancia_explicada_2d)
figura_dispersao_por_ano.show()

In [15]:
resultado = montar_tabela_pertencimento(matriz.index, rotulos)

figura_regional = grafico_distribuicao_por_regional(resultado)
figura_regional.show()

figura_ano = grafico_distribuicao_por_ano(resultado)
figura_ano.show()

### Composição de cada cluster (tabela, não só o gráfico)

A dispersão acima projeta em PC1 x PC2 -- dois clusters que o KMeans separou no espaço original podem aparecer sobrepostos no 2D. Esta tabela lista, por CLUSTER, quantos pares e quais Regional-Ano pertencem a ele, direto dos rótulos do KMeans, sem depender da projeção. Diferente do DBSCAN, não há rótulo de ruído -- todo par pertence a algum cluster.

In [16]:
resumo_por_cluster = montar_resumo_por_cluster(resultado)
resumo_por_cluster

,CLUSTER,N_PARES,N_REGIONAIS_DISTINTAS,MEMBROS
0,0,169,23,"alfenas (2008), alfenas (2009), alfenas (2011)..."
1,1,3,2,"coronel fabriciano (2009), coronel fabriciano ..."
2,2,51,15,"alfenas (2016), barbacena (2014), barbacena (2..."
3,3,120,27,"alfenas (2003), alfenas (2005), alfenas (2007)..."
4,4,28,13,"alfenas (2014), alfenas (2015), alfenas (2017)..."


## Saídas finais

In [17]:
resumo_parametros = montar_resumo_parametros_kmeans(
    k_escolhido, inercia_final, silhueta_final,
    usar_robust_scaler=True, pca=pca, variancia_acumulada=variancia_acumulada,
    n_linhas=matriz.shape[0], n_atributos=matriz.shape[1], n_clusters=n_clusters,
)
resumo_parametros

,K,ESCALONADOR,PCA_APLICADO,PCA_N_COMPONENTES,PCA_VARIANCIA_EXPLICADA,N_LINHAS,N_ATRIBUTOS,N_CLUSTERS,INERCIA,SILHUETA
0,5,RobustScaler,True,9,0.867563,371,22,5,2609.254173,0.211154


In [18]:
atributos_usados = pd.DataFrame({"ATRIBUTO": list(matriz.columns)})
atributos_usados

,ATRIBUTO
0,casos-faixa_10-19
1,casos-faixa_20-29
2,casos-faixa_30-39
3,casos-faixa_40-49
4,casos-faixa_50-59
5,casos-faixa_60-69
6,casos-faixa_70-ou-mais
7,casos-escolaridade_fundamental-completo
8,casos-escolaridade_fundamental-incompleto
9,casos-escolaridade_nenhuma


In [19]:
resultado

,REGIONAL DE SAÚDE,ANO,CLUSTER
0,alfenas,2003,3
1,alfenas,2005,3
2,alfenas,2007,3
3,alfenas,2008,0
4,alfenas,2009,0
...,...,...,...
366,varginha,2018,2
367,varginha,2019,2
368,varginha,2020,0
369,varginha,2021,0


In [20]:
visao_agregada_regional_ano = montar_visao_agregada_regional_ano(resultado)
visao_agregada_regional_ano

ANO,2000,2003,2005,2007,2008,2009,2010,2011,2012,2014,2015,2016,2017,2018,2019,2020,2021,2023
REGIONAL DE SAÚDE,,,,,,,,,,,,,,,,,,
alfenas,NaN,3.0,3.0,3.0,0.0,0.0,NaN,0.0,0.0,4.0,4.0,2.0,4.0,4.0,0.0,4.0,4.0,4.0
barbacena,3.0,3.0,3.0,NaN,0.0,NaN,0.0,NaN,0.0,2.0,NaN,0.0,2.0,2.0,2.0,0.0,NaN,NaN
belo horizonte,3.0,3.0,3.0,0.0,0.0,2.0,0.0,4.0,0.0,0.0,0.0,2.0,2.0,0.0,0.0,0.0,0.0,4.0
coronel fabriciano,NaN,3.0,3.0,3.0,3.0,1.0,1.0,NaN,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
diamantina,NaN,NaN,NaN,3.0,NaN,3.0,3.0,3.0,NaN,3.0,3.0,3.0,NaN,3.0,0.0,NaN,3.0,4.0
divinopolis,3.0,3.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0
governador valadares,3.0,3.0,3.0,3.0,3.0,4.0,1.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0
itabira,NaN,NaN,NaN,0.0,3.0,0.0,0.0,0.0,3.0,0.0,3.0,0.0,0.0,3.0,0.0,0.0,0.0,2.0
ituiutaba,NaN,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,4.0,0.0,NaN,NaN,0.0,3.0,0.0,NaN


### Heatmap: cluster por Regional e Ano

Mesma informação de `visao_agregada_regional_ano`, mas como cor em vez de número -- uma cor por CLUSTER, célula em branco onde não há dado (par descartado no dropna ou fora da cobertura de algum bloco).

In [21]:
figura_heatmap = grafico_heatmap_regional_ano(visao_agregada_regional_ano)
figura_heatmap.show()